In [1]:
import pandas as pd
import numpy as np

from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")


# ==========================================
# 1. Load Dataset
# ==========================================

df = pd.read_csv(
    "data/market_data_tda_pelt_fixed.csv",
    parse_dates=["Date"],
    index_col="Date"
)

print("Dataset Shape:", df.shape)


# ==========================================
# 2. HMM Configuration
# ==========================================

hmm_features = [

    "SP500_Return",

    "SP500_Volatility_20",

    "VIX",

    "TDA_H1_TotalPersistence",

    "TDA_H1_Entropy",

    "PELT_Structural_Stress"
]


N_STATES = 3

MIN_TRAIN = 500

REFIT_EVERY = 60


# ==========================================
# 3. Output
# ==========================================

df["HMM_State"] = np.nan


print(
    "\nRunning Causal Expanding HMM..."
)


# ==========================================
# 4. Expanding Window HMM
# ==========================================

model = None
scaler = None

for i in range(
    MIN_TRAIN,
    len(df)
):

    # --------------------------------------
    # Refit periodically using ONLY past
    # --------------------------------------

    if (
        model is None
        or
        (i - MIN_TRAIN) % REFIT_EVERY == 0
    ):

        train = df.iloc[
            :i
        ][hmm_features].copy()


        # Standardize using past only

        scaler = StandardScaler()

        X_train = scaler.fit_transform(
            train
        )


        # HMM

        model = GaussianHMM(

            n_components=N_STATES,

            covariance_type="diag",

            n_iter=200,

            random_state=42

        )

        model.fit(
            X_train
        )


    # --------------------------------------
    # Predict current state
    # --------------------------------------

    # Use recent sequence so HMM has context

    sequence_start = max(
        0,
        i - 59
    )

    sequence = df.iloc[
        sequence_start:i + 1
    ][hmm_features]


    X_sequence = scaler.transform(
        sequence
    )


    states = model.predict(
        X_sequence
    )


    df.iloc[
        i,
        df.columns.get_loc(
            "HMM_State"
        )
    ] = states[-1]


    # Progress

    if (
        i - MIN_TRAIN + 1
    ) % 500 == 0:

        print(
            f"Processed "
            f"{i - MIN_TRAIN + 1}/"
            f"{len(df) - MIN_TRAIN}"
        )


# ==========================================
# 5. Analyze Raw States
# ==========================================

valid = df.dropna(
    subset=["HMM_State"]
).copy()

valid["HMM_State"] = (
    valid["HMM_State"]
    .astype(int)
)


print(
    "\nRaw HMM State Distribution:"
)

print(
    valid[
        "HMM_State"
    ].value_counts()
)


print(
    "\nState Characteristics:"
)

state_stats = valid.groupby(
    "HMM_State"
)[
    [
        "SP500_Return",
        "SP500_Volatility_20",
        "VIX",
        "Crash_Risk"
    ]
].mean()

print(
    state_stats
)


# ==========================================
# 6. Rank States by Market Stress
# ==========================================

# Do NOT assume:
# State 0 = Stable
# State 1 = Volatile
# State 2 = Crisis
#
# HMM state IDs are arbitrary.

state_stats[
    "Stress_Score"
] = (

    state_stats[
        "SP500_Volatility_20"
    ].rank()

    +

    state_stats[
        "VIX"
    ].rank()

    +

    state_stats[
        "Crash_Risk"
    ].rank()

)


ordered_states = (

    state_stats[
        "Stress_Score"
    ]

    .sort_values()

    .index

    .tolist()

)


regime_mapping = {

    ordered_states[0]:
    "Stable",

    ordered_states[1]:
    "Volatile",

    ordered_states[2]:
    "Crisis"

}


print(
    "\nRegime Mapping:"
)

print(
    regime_mapping
)


# ==========================================
# 7. Create Regime Column
# ==========================================

df["Market_Regime"] = (

    df["HMM_State"]
    .map(
        regime_mapping
    )

)


# ==========================================
# 8. Create Numeric Regime Risk
# ==========================================

risk_mapping = {

    "Stable": 0,

    "Volatile": 1,

    "Crisis": 2

}

df["HMM_Regime_Risk"] = (

    df["Market_Regime"]
    .map(
        risk_mapping
    )

)


# ==========================================
# 9. Validation
# ==========================================

print(
    "\nMarket Regime Distribution:"
)

print(
    df[
        "Market_Regime"
    ].value_counts()
)


print(
    "\nCrash Rate by Regime:"
)

print(

    df.dropna(
        subset=[
            "Market_Regime"
        ]
    )

    .groupby(
        "Market_Regime"
    )[
        "Crash_Risk"
    ]

    .agg(
        [
            "count",
            "sum",
            "mean"
        ]
    )

)


# ==========================================
# 10. Known Crisis Check
# ==========================================

for name, period in {

    "Financial Crisis":
    ("2008-09-01", "2009-03-31"),

    "COVID Crash":
    ("2020-02-01", "2020-04-30")

}.items():

    start, end = period

    print(
        f"\n{name}:"
    )

    print(

        df.loc[
            start:end
        ][
            [
                "SP500",
                "VIX",
                "Crash_Risk",
                "Market_Regime"
            ]
        ]

        .head(30)

    )


# ==========================================
# 11. Save
# ==========================================

df.to_csv(
    "data/market_data_tda_pelt_hmm.csv"
)

print(
    "\nFinal Shape:",
    df.shape
)

print(
    "\nSaved:"
    " data/market_data_tda_pelt_hmm.csv"
)

Dataset Shape: (6423, 45)

Running Causal Expanding HMM...


Model is not converging.  Current: -5003.040281196848 is not greater than -5002.821756657764. Delta is -0.21852453908377356


Processed 500/5923
Processed 1000/5923
Processed 1500/5923
Processed 2000/5923
Processed 2500/5923
Processed 3000/5923
Processed 3500/5923
Processed 4000/5923
Processed 4500/5923
Processed 5000/5923
Processed 5500/5923

Raw HMM State Distribution:
HMM_State
1    2113
2    2091
0    1719
Name: count, dtype: int64

State Characteristics:
           SP500_Return  SP500_Volatility_20        VIX  Crash_Risk
HMM_State                                                          
0              0.000411             0.138251  18.188406    0.017452
1              0.000185             0.160883  19.378632    0.053005
2              0.000713             0.163978  19.607394    0.033955

Regime Mapping:
{0: 'Stable', 1: 'Volatile', 2: 'Crisis'}

Market Regime Distribution:
Market_Regime
Volatile    2113
Crisis      2091
Stable      1719
Name: count, dtype: int64

Crash Rate by Regime:
               count  sum      mean
Market_Regime                      
Crisis          2091   71  0.033955
Stable      

In [2]:
import pandas as pd

df = pd.read_csv(
    "data/market_data_tda_pelt_hmm.csv",
    parse_dates=["Date"],
    index_col="Date"
)

valid = df.dropna(
    subset=["HMM_State"]
).copy()

valid["HMM_State"] = (
    valid["HMM_State"].astype(int)
)

# ------------------------------------------
# Target-independent regime characteristics
# ------------------------------------------

state_stats = valid.groupby(
    "HMM_State"
)[
    [
        "SP500_Volatility_20",
        "VIX"
    ]
].mean()

# Rank by market stress
state_stats["Stress_Score"] = (
    state_stats["SP500_Volatility_20"].rank()
    +
    state_stats["VIX"].rank()
)

ordered_states = (
    state_stats["Stress_Score"]
    .sort_values()
    .index
    .tolist()
)

regime_mapping = {
    ordered_states[0]: "Stable",
    ordered_states[1]: "Volatile",
    ordered_states[2]: "Crisis"
}

print("\nState Statistics:")
print(state_stats)

print("\nCorrected Regime Mapping:")
print(regime_mapping)

# ------------------------------------------
# Replace old mapping
# ------------------------------------------

df["Market_Regime"] = (
    df["HMM_State"]
    .map(regime_mapping)
)

risk_mapping = {
    "Stable": 0,
    "Volatile": 1,
    "Crisis": 2
}

df["HMM_Regime_Risk"] = (
    df["Market_Regime"]
    .map(risk_mapping)
)

# ------------------------------------------
# Evaluation only — Crash_Risk is NOT
# used to create the mapping
# ------------------------------------------

print("\nCrash Rate by Corrected Regime:")

print(
    df.dropna(subset=["Market_Regime"])
    .groupby("Market_Regime")["Crash_Risk"]
    .agg(["count", "sum", "mean"])
)

df.to_csv(
    "data/market_data_final_features.csv"
)

print("\nFinal Shape:", df.shape)
print(
    "Saved: data/market_data_final_features.csv"
)


State Statistics:
           SP500_Volatility_20        VIX  Stress_Score
HMM_State                                              
0                     0.138251  18.188406           2.0
1                     0.160883  19.378632           4.0
2                     0.163978  19.607394           6.0

Corrected Regime Mapping:
{0: 'Stable', 1: 'Volatile', 2: 'Crisis'}

Crash Rate by Corrected Regime:
               count  sum      mean
Market_Regime                      
Crisis          2091   71  0.033955
Stable          1719   30  0.017452
Volatile        2113  112  0.053005

Final Shape: (6423, 48)
Saved: data/market_data_final_features.csv
